## CHILI DATA - DAY 03 - EUROSTAT API  

In [3]:
# ES Api code here! --> A - on it! TODO

# Infos on ES & CT API USE and DS-045409 dataset - Endpoints, Output limits - google

The DS-045409 dataset (EU trade since 1988 by HS2-4-6 and CN8) is a detailed trade database accessible via Eurostat's Comext SDMX REST API and related tools. 
It provides monthly and annual data by individual EU reporter, partner country, and product code. 

Accessing DS-045409 Trade Data

Eurostat Comext API: Data can be accessed via the SDMX 2.1 REST API, which enables automated downloads.
        Base URL: https://ec.europa.eu/eurostat/api/comext/dissemination/sdmx/2.1/dataflow/ESTAT/all
    R Programming (restatapi or comRex): The restatapi package can be used for automation. Additionally, the comRex R package provides a wrapper specifically to access Comext data with ds_id="045409".
    EasyComext (Manual/Batch): DS-045409 is part of the EasyComext user interface, which allows for setting up to 6 parameters for data extraction, including reporter, partner, product, and flow.
    Bulk Download: Eurostat provides a bulk download facility that offers the data in CSV format, with datasets updated regularly to the latest reference month. 

Key Data Features

    Coverage: Detailed data for EU member states, including CN8 (Combined Nomenclature 8-digit) codes.
    Time Period: Monthly and annual trade statistics from 1988 onwards.
    Timezone: It is recommended to define timezones (e.g., UTC, CET) when accessing data through the API to ensure accurate timing of data updates. 

Technical Considerations

    Parameter Updating: The dataset ID can be indicated in query files using the <ds-extid> tag, particularly when transitioning from older datasets.
    Location Changes: The URL structure for Eurostat datasets changes frequently; monitoring the API documentation is advised. 

# Infos on CIF (Standard for IMPORT totals) and FOB (EXPORT) values - AI guided

In the world of trade data, CIF and FOB are "Incoterms" (International Commercial Terms) that define exactly what is included in the dollar value you are seeing in your database.

When you look at trade data for Fresh Capsicum (Bell Peppers), these values tell you whether the "price tag" includes the cost of getting the peppers across the ocean or just the cost of the peppers themselves at the loading dock.

1. CIF Value (Cost, Insurance, and Freight)
This is almost always the value used for Import totals.
    What it represents: The value of the goods plus everything required to get them to the border of the importing country.
    The Formula: CIF=Value of Peppers+Shipping/Freight+Insurance
    Why it matters: If you are analyzing "Import Revenue" or "Import Value," CIF is the "landed" cost. It’s what the importing country actually paid to have that product arrive at their port.

2. FOB Value (Free On Board)
This is the standard value used for Export totals.
    What it represents: The value of the goods at the moment they were loaded onto the ship/truck in the country of origin. It excludes international shipping and insurance.
    The Formula: FOB=Value of Peppers+Cost to load them
    Why it matters: FOB reflects the "true" domestic value of the product for the exporter. It’s the revenue that actually stays in the exporting country’s economy before shipping companies (who might be from a third country) take their cut.

3. The "Capsicum" Context (HS Code 0709.60)
For Fresh Bell Peppers, the distinction is vital because they are perishable.

    HS Code: The base code for fresh/chilled fruits of the genus Capsicum is 0709.60.
    Specifics: You will likely see 0709.60.10 for "Sweet Peppers" (Bell Peppers).

    Data Analysis Gap: If you compare an export record from Mexico (FOB) to an import record in the USA (CIF), the numbers will never match. The USA value will be higher because it includes the cost of the refrigerated trucks and insurance needed to keep the peppers fresh during transit.
    Pro-Tip for your Data Wrangling: If you see a column called cifvalue and another called quantity_kg, dividing them gives you the Unit Price including shipping. If you want to know the "farm-gate" price, you would need the FOB value.


## COMTRADE DS ONLY 

In [ ]:
# MY CODE HERE !!! 

In [ ]:
# FIRST TRY - COMTRADE API - get HS datasets to make Main DF CT_df --> A 

def get_CT_data(reporter_codes, cmd_codes, years):
    """ Fetch trade data using the UN Comtrade Public Preview API."""

    # The 'preview' endpoint is the one that works without subscription key
    base_url = f"https://comtradeapi.un.org/public/v1/preview/C/A/HS"
    
    params = {
        'reporterCode': reporter_codes, 
        'period': years,              
        'cmdCode': cmd_codes,         
        'flowCode': 'M',                # M = Imports
        'partnerCode': '0',             # 0 = World
        'format': 'JSON'
    }
    
    print(f"Requesting data for codes: {cmd_codes}...")
    response = requests.get(base_url, params=params)
    
    if response.status_code == 200:
        data = response.json()
        if 'data' in data and data['data']:
            return pd.DataFrame(data['data'])
        else:
            print("No data found for these parameters.")
            return None
    else:
        print(f"Error {response.status_code}: {response.text}")
        return None

# --- EXECUTION ---

# DE=276, UK=826, PL=616 
country_codes = '276,826,616'
# HS Codes: Fresh Chili, Dried/Crushed, Hot Sauces, Preserves 
# HS 210390 = Sauces/Condiments; 200190 = Pickled/Preserved (Harissa)
hs_codes = '070960,090421,090422,210390,200190'

years = '2020,2021,2022,2023,2024,2025' 

CT_api_df = get_CT_data(country_codes, hs_codes, years)

if CT_api_df is not None:
    # 1. The complete DF
    print("Master Dataset Head:\n", CT_api_df.head())

    # 2. Split into individual country DFs using 'reporterCode'
    # Note: Comtrade returns numeric codes as ints or strings; ensure consistency
    df_de = CT_api_df[CT_api_df['reporterCode'].astype(str) == '276']
    df_uk = CT_api_df[CT_api_df['reporterCode'].astype(str) == '826']
    df_pl = CT_api_df[CT_api_df['reporterCode'].astype(str) == '616']
    
    print(f"\nExtracted: DE({len(df_de)} rows), UK({len(df_uk)} rows), PL({len(df_pl)} rows)")

## COMTRADE CSV & JSON Data Gathering - MYKYTA

--> ALL - MERGE NOTEBOOKS/REPO DAY4 - TODO

## COMTRADE IMPLIED Varieties INFO --> ALL 

# Different Info scraping options, explore if we have time! 

# Careful with GENERATED Data - Check AI output --> Miguel Comment: Just be explicit about your research process!
    Use implied, categorical data about Varieties, Reporter Countries more as a "heuristic", just remember it is not quantitative but categorical Analysis!
# Same with mapping Varieties to countries of origins - also QUALITATIVE Method but we get a nice Chili Info db with our own data 


## Plotted graphs in GH repo & Slack 
--> M - Organize Data json & csv OUTPUTS! - TODO